# Deploy a Durable Multi-Agent Workflow

This notebook runs a durable multi-agent system where:

- Docker Compose runs the Agentspan server, Postgres, and four sandbox APIs.
- Python runs outside Docker as the SDK client and worker process.
- Agentspan coordinates a multi-agent workflow with parallel lanes.
- A billing API outage blocks one lane while other lanes finish.
- When billing comes back, the same execution continues and completes.



## What Agentspan Is Showing Here

Agentspan is an execution layer for AI agents. You define agents and tools in Python, while execution state lives on the Agentspan server. That gives each run a stable execution ID, server-side task history, child-agent executions, approval waits, and a UI trail.

This demo treats an agent system as distributed software. The workflow touches separate services, runs three child agents in parallel, records every tool boundary, pauses around risky work, and keeps the execution inspectable when an external dependency is unavailable.

## 1. Install Prerequisites

This notebook uses Docker Compose. In a local Jupyter environment or VM, Docker should already be installed. In Colab, the setup cell attempts to install Docker and start a Docker daemon. If your Colab runtime blocks Docker daemon startup, run the same notebook on a local machine or VM with Docker enabled.

The default model is `openai/gpt-5.5`. You can change `AGENTSPAN_LLM_MODEL` to any supported provider/model and set that provider's environment variables.

In [ ]:
import getpass
import os
import shutil
import subprocess
import sys
import time
from importlib.metadata import version
from pathlib import Path

from IPython.display import HTML, display


ROOT = Path("/content/agentspan-docker-e2e")
ROOT.mkdir(parents=True, exist_ok=True)
AGENTSPAN_VERSION = "0.1.10"


def sh(cmd: str, *, check: bool = True, timeout: int | None = None, cwd: Path | None = None):
    print(f"$ {cmd}")
    result = subprocess.run(
        cmd,
        shell=True,
        executable="/bin/bash",
        text=True,
        capture_output=True,
        timeout=timeout,
        cwd=str(cwd or ROOT),
        env=os.environ.copy(),
    )
    if result.stdout:
        print(result.stdout.strip())
    if result.stderr:
        print(result.stderr.strip())
    if check and result.returncode != 0:
        raise RuntimeError(f"command failed: {cmd}")
    return result


os.environ.setdefault("AGENTSPAN_LLM_MODEL", "openai/gpt-5.5")
if os.environ["AGENTSPAN_LLM_MODEL"].startswith("openai/") and not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")

# Java is useful for local CLI/server commands. Docker runs the server in this demo.
sh("apt-get update -qq", check=False, timeout=300, cwd=Path("/"))
sh("apt-get install -y -qq openjdk-21-jre-headless", check=False, timeout=600, cwd=Path("/"))
sh(f"{sys.executable} -m pip install -qU agentspan=={AGENTSPAN_VERSION} requests", timeout=600)

print("python_agentspan_version =", version("agentspan"))
print("model =", os.environ["AGENTSPAN_LLM_MODEL"])
print("working_dir =", ROOT)

In [ ]:
def docker_ok() -> bool:
    return subprocess.run(
        "docker info",
        shell=True,
        executable="/bin/bash",
        text=True,
        capture_output=True,
    ).returncode == 0


if not shutil.which("docker"):
    sh("apt-get install -y -qq docker.io docker-compose-plugin", timeout=600, cwd=Path("/"))

if not docker_ok():
    print("starting Docker daemon")
    subprocess.Popen(
        "nohup dockerd --host=unix:///var/run/docker.sock --storage-driver=vfs > /tmp/dockerd.log 2>&1 &",
        shell=True,
        executable="/bin/bash",
    )
    for _ in range(60):
        if docker_ok():
            break
        time.sleep(2)
    else:
        print(Path("/tmp/dockerd.log").read_text()[-4000:] if Path("/tmp/dockerd.log").exists() else "no dockerd log")
        raise RuntimeError("Docker did not become available in this runtime.")

sh("docker version --format '{{.Server.Version}}'", timeout=60)
sh("docker compose version", timeout=60)

## 2. Define the Docker Stack

The Docker stack has:

- `postgres`: persistent state for the Agentspan server.
- `agentspan`: the Agentspan server and UI, exposed on port `6797`.
- `crm-api`, `identity-api`, `product-api`, `billing-api`: sandbox business systems.

The Python SDK and worker run outside Docker. That separation is intentional: it shows the server/runtime boundary clearly.

In [ ]:
(ROOT / "docker-compose.yml").write_text("""name: agentspan-e2e-notebook

services:
  postgres:
    image: postgres:16-alpine
    environment:
      POSTGRES_USER: agentspan
      POSTGRES_PASSWORD: changeme
      POSTGRES_DB: agentspan
    ports:
      - "5547:5432"
    volumes:
      - postgres_data:/var/lib/postgresql/data
    healthcheck:
      test: ["CMD-SHELL", "pg_isready -U agentspan -d agentspan"]
      interval: 5s
      timeout: 3s
      retries: 20

  agentspan:
    image: agentspan/server:0.1.10
    depends_on:
      postgres:
        condition: service_healthy
    ports:
      - "6797:6767"
    environment:
      SPRING_PROFILES_ACTIVE: postgres
      SPRING_DATASOURCE_URL: jdbc:postgresql://postgres:5432/agentspan
      SPRING_DATASOURCE_USERNAME: agentspan
      SPRING_DATASOURCE_PASSWORD: changeme
      OPENAI_API_KEY: ${OPENAI_API_KEY:-}
      JAVA_TOOL_OPTIONS: -Xms512m -Xmx1536m -XX:+UseG1GC
      LOGGING_LEVEL_ROOT: WARN
      LOGGING_LEVEL_DEV_AGENTSPAN: INFO

  crm-api:
    image: python:3.12-slim
    working_dir: /app
    command: python sandbox_services.py crm
    ports:
      - "18111:8011"
    volumes:
      - .:/app:ro

  billing-api:
    image: python:3.12-slim
    working_dir: /app
    command: python sandbox_services.py billing
    ports:
      - "18112:8012"
    volumes:
      - .:/app:ro

  identity-api:
    image: python:3.12-slim
    working_dir: /app
    command: python sandbox_services.py identity
    ports:
      - "18113:8013"
    volumes:
      - .:/app:ro

  product-api:
    image: python:3.12-slim
    working_dir: /app
    command: python sandbox_services.py product
    ports:
      - "18114:8014"
    volumes:
      - .:/app:ro

volumes:
  postgres_data:
""")

print((ROOT / "docker-compose.yml").read_text())

## 3. Create the Sandbox APIs

These are intentionally small services. They make the demo concrete without calling real customer, billing, identity, or product systems.

The billing service is the dependency we will stop and restart during the run.

In [ ]:
(ROOT / "sandbox_services.py").write_text(r"""#!/usr/bin/env python3
from __future__ import annotations

import json
import os
import sys
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from urllib.parse import parse_qs, urlparse

DATA_DIR = Path("/tmp/agentspan-e2e-services")
DATA_DIR.mkdir(parents=True, exist_ok=True)


def send_json(handler: BaseHTTPRequestHandler, status: int, payload: dict) -> None:
    body = json.dumps(payload, indent=2).encode("utf-8")
    handler.send_response(status)
    handler.send_header("content-type", "application/json")
    handler.send_header("content-length", str(len(body)))
    handler.end_headers()
    handler.wfile.write(body)


class ServiceHandler(BaseHTTPRequestHandler):
    service = "unknown"

    def log_message(self, fmt: str, *args: object) -> None:
        sys.stdout.write(f"[{self.service}] {fmt % args}\n")
        sys.stdout.flush()

    def do_GET(self) -> None:
        parsed = urlparse(self.path)
        path = parsed.path
        query = parse_qs(parsed.query)

        if path == "/health":
            return send_json(self, 200, {"service": self.service, "status": "ok"})

        if self.service == "crm" and path == "/customers":
            email = query.get("email", ["alice@example.com"])[0]
            return send_json(self, 200, {
                "customer_id": "cust_enterprise_001",
                "email": email,
                "name": "Alice Chen",
                "tier": "enterprise",
                "account_status": "active",
                "contract": "enterprise-onboarding",
            })

        if self.service == "identity" and path.startswith("/users/") and path.endswith("/permissions"):
            email = path.split("/")[2]
            return send_json(self, 200, {
                "email": email,
                "role": "admin",
                "mfa_enabled": True,
                "can_create_workspace": True,
                "can_enable_paid_plan": True,
            })

        if self.service == "billing" and path.startswith("/accounts/") and path.endswith("/billing-status"):
            customer_id = path.split("/")[2]
            return send_json(self, 200, {
                "customer_id": customer_id,
                "plan": "trial",
                "eligible_for_enterprise": True,
                "unpaid_invoices": 0,
                "requires_approval": True,
            })

        return send_json(self, 404, {"error": "not_found", "path": path})

    def do_POST(self) -> None:
        parsed = urlparse(self.path)
        length = int(self.headers.get("content-length", "0"))
        raw = self.rfile.read(length) if length else b"{}"
        try:
            body = json.loads(raw.decode("utf-8"))
        except json.JSONDecodeError:
            body = {}

        if self.service == "product" and parsed.path == "/workspaces":
            key = self.headers.get("Idempotency-Key", "missing-key")
            state_file = DATA_DIR / "workspaces.json"
            state = json.loads(state_file.read_text()) if state_file.exists() else {}
            if key not in state:
                state[key] = {
                    "workspace_id": "ws_enterprise_001",
                    "customer_id": body.get("customer_id"),
                    "status": "created",
                    "idempotency_key": key,
                }
                state_file.write_text(json.dumps(state, indent=2))
            return send_json(self, 200, state[key])

        if self.service == "billing" and parsed.path.startswith("/accounts/") and parsed.path.endswith("/plan"):
            customer_id = parsed.path.split("/")[2]
            return send_json(self, 200, {
                "customer_id": customer_id,
                "plan": body.get("plan", "enterprise"),
                "status": "enabled",
                "approval_checked": True,
            })

        return send_json(self, 404, {"error": "not_found", "path": parsed.path})


def run(service: str, port: int) -> None:
    handler = type(f"{service.title()}Handler", (ServiceHandler,), {"service": service})
    server = ThreadingHTTPServer(("0.0.0.0", port), handler)
    print(f"{service}-api listening on {port}", flush=True)
    server.serve_forever()


if __name__ == "__main__":
    ports = {"crm": 8011, "billing": 8012, "identity": 8013, "product": 8014}
    selected = sys.argv[1] if len(sys.argv) > 1 else os.environ.get("SERVICE")
    if selected not in ports:
        raise SystemExit(f"usage: {sys.argv[0]} [{'|'.join(ports)}]")
    run(selected, ports[selected])
""")

print("wrote", ROOT / "sandbox_services.py")

## 4. Start the Stack

This is the first thing to look at in the demo: the system is not one Python process. The server, database, and external services are separate processes with their own failure modes.

In [ ]:
sh("docker compose down -v --remove-orphans", check=False, timeout=120)
sh("docker compose up -d", timeout=900)
sh("docker compose ps", timeout=120)

SERVER_LOCAL = "http://localhost:6797"
SERVER_URL = f"{SERVER_LOCAL}/api"
os.environ["AGENTSPAN_SERVER_URL"] = SERVER_URL

try:
    from google.colab import output

    UI_ROOT = output.eval_js("google.colab.kernel.proxyPort(6797)").rstrip("/")
except Exception:
    UI_ROOT = SERVER_LOCAL

print("ui_root =", UI_ROOT)
display(HTML(f'<a href="{UI_ROOT}" target="_blank">Open Agentspan UI</a>'))

## 5. Define Agentspan Tools and Agents

The `@tool` functions are the integration boundary. The agent can reason about what to do, but tool calls are where the workflow touches external systems.

There are three child agents:

- CRM reviewer
- Security reviewer
- Billing/provisioning lane

The parent agent uses a parallel strategy so those child lanes can run independently.

In [ ]:
(ROOT / "onboarding_system.py").write_text(r'''#!/usr/bin/env python3
from __future__ import annotations

import os
import subprocess
import sys
import time
from pathlib import Path
from typing import Any

import requests
from agentspan.agents import Agent, AgentHandle, AgentRuntime, Strategy, ToolContext, tool

ROOT = Path(__file__).resolve().parent
SERVER_URL = os.environ.get("AGENTSPAN_SERVER_URL", "http://localhost:6797/api")
UI_ROOT = os.environ.get("AGENTSPAN_UI_ROOT", "http://localhost:6797")
MODEL = os.environ.get("AGENTSPAN_LLM_MODEL", "openai/gpt-5.5")
REGISTERED_FILE = ROOT / "registered_name.txt"
EXECUTION_FILE = ROOT / "latest_execution_id.txt"

CRM_URL = "http://localhost:18111"
BILLING_URL = "http://localhost:18112"
IDENTITY_URL = "http://localhost:18113"
PRODUCT_URL = "http://localhost:18114"


def request_json(method: str, url: str, **kwargs: Any) -> dict:
    response = requests.request(method, url, timeout=4, **kwargs)
    response.raise_for_status()
    return response.json()


@tool
def lookup_customer(email: str) -> dict:
    'Look up the customer record in the CRM service.'
    return request_json("GET", f"{CRM_URL}/customers", params={"email": email})


@tool
def verify_requester(email: str) -> dict:
    'Verify requester permissions in the identity service.'
    safe_email = email.replace("@", "%40")
    return request_json("GET", f"{IDENTITY_URL}/users/{safe_email}/permissions")


@tool
def prepare_billing_review(customer_id: str) -> dict:
    'Mark the point where billing/provisioning is about to begin.'
    return {"customer_id": customer_id, "checkpoint": "ready_for_billing_review"}


@tool(timeout_seconds=240)
def check_billing_status(customer_id: str) -> dict:
    'Check billing eligibility, retrying while the billing service is unavailable.'
    last_error = ""
    for attempt in range(1, 121):
        try:
            result = request_json("GET", f"{BILLING_URL}/accounts/{customer_id}/billing-status")
            result["dependency_recovered_after_attempts"] = attempt
            return result
        except requests.RequestException as exc:
            last_error = str(exc)
            print(f"billing_status_attempt={attempt} dependency_unreachable")
            time.sleep(2)
    raise RuntimeError(f"billing service did not recover: {last_error}")


@tool
def create_workspace(customer_id: str, context: ToolContext) -> dict:
    'Create a workspace with a stable idempotency key.'
    execution_id = getattr(context, "workflow_id", None) or getattr(context, "execution_id", "unknown")
    key = f"{execution_id}:create_workspace:{customer_id}"
    return request_json(
        "POST",
        f"{PRODUCT_URL}/workspaces",
        json={"customer_id": customer_id},
        headers={"Idempotency-Key": key},
    )


@tool(approval_required=True)
def enable_enterprise_plan(customer_id: str, plan: str = "enterprise") -> dict:
    'Enable a paid plan after a human approval checkpoint.'
    return request_json("POST", f"{BILLING_URL}/accounts/{customer_id}/plan", json={"plan": plan})


def build_workflow() -> Agent:
    crm_agent = Agent(
        name="notebook_crm_reviewer",
        model=MODEL,
        tools=[lookup_customer],
        temperature=0,
        max_turns=4,
        instructions="Call lookup_customer for alice@example.com. Return customer id, tier, and status.",
    )

    security_agent = Agent(
        name="notebook_security_reviewer",
        model=MODEL,
        tools=[verify_requester],
        temperature=0,
        max_turns=4,
        instructions="Call verify_requester for ops.admin@example.com. Return whether onboarding is allowed.",
    )

    provisioning_agent = Agent(
        name="notebook_billing_provisioning_lane",
        model=MODEL,
        tools=[prepare_billing_review, check_billing_status, create_workspace, enable_enterprise_plan],
        temperature=0,
        max_turns=10,
        instructions=(
            "Use tools in this exact order: prepare_billing_review for cust_enterprise_001, "
            "check_billing_status for cust_enterprise_001, create_workspace for cust_enterprise_001, "
            "then enable_enterprise_plan for cust_enterprise_001 with plan enterprise. "
            "The billing API may be temporarily unreachable; keep the billing lane pending until it recovers."
        ),
    )

    return Agent(
        name="notebook_enterprise_onboarding_parallel_demo",
        model=MODEL,
        agents=[crm_agent, security_agent, provisioning_agent],
        strategy=Strategy.PARALLEL,
        instructions="Run CRM, security, and billing/provisioning lanes in parallel, then summarize their results.",
    )


def workflow_json(execution_id: str) -> dict:
    base = SERVER_URL.removesuffix("/api")
    return request_json("GET", f"{base}/api/workflow/{execution_id}")


def walk_tasks(execution_id: str, seen: set[str] | None = None) -> list[dict]:
    seen = seen or set()
    if execution_id in seen:
        return []
    seen.add(execution_id)
    workflow = workflow_json(execution_id)
    tasks = list(workflow.get("tasks", []))
    for task in list(tasks):
        child_id = task.get("subWorkflowId") or task.get("outputData", {}).get("subWorkflowId")
        if child_id:
            tasks.extend(walk_tasks(child_id, seen))
    return tasks


def task_completed(execution_id: str, fragment: str) -> bool:
    for task in walk_tasks(execution_id):
        names = [str(task.get(key) or "") for key in ("referenceTaskName", "taskDefName", "taskType")]
        if any(fragment in name for name in names) and task.get("status") == "COMPLETED":
            return True
    return False


def task_active(execution_id: str, fragment: str) -> bool:
    for task in walk_tasks(execution_id):
        names = [str(task.get(key) or "") for key in ("referenceTaskName", "taskDefName", "taskType")]
        if any(fragment in name for name in names) and task.get("status") in {"IN_PROGRESS", "SCHEDULED"}:
            return True
    return False


def waiting_human_execution_id(execution_id: str) -> str | None:
    for task in walk_tasks(execution_id):
        if task.get("taskType") == "HUMAN" and task.get("status") in {"IN_PROGRESS", "SCHEDULED"}:
            return task.get("workflowInstanceId")
    return None


def approve_waiting_human(runtime: AgentRuntime, execution_id: str) -> bool:
    target = waiting_human_execution_id(execution_id)
    if not target:
        return False
    AgentHandle(execution_id=target, runtime=runtime).approve()
    return True


def prompt_text() -> str:
    return (
        "Enterprise onboarding request: provision alice@example.com for enterprise onboarding. "
        "Requester is ops.admin@example.com. Create the workspace and enable the enterprise plan if checks pass."
    )


def deploy() -> None:
    with AgentRuntime(server_url=SERVER_URL) as runtime:
        info = runtime.deploy(build_workflow())[0]
        REGISTERED_FILE.write_text(info.registered_name + "\n")
        print("registered_name =", info.registered_name)


def serve() -> None:
    with AgentRuntime(server_url=SERVER_URL) as runtime:
        print("worker_service = polling for Python tool tasks")
        runtime.serve(build_workflow())


def start_execution() -> None:
    registered_name = REGISTERED_FILE.read_text().strip()
    with AgentRuntime(server_url=SERVER_URL) as runtime:
        handle = runtime.start(registered_name, prompt_text())
        EXECUTION_FILE.write_text(handle.execution_id + "\n")
        print("execution_id =", handle.execution_id)
        print("ui =", f"{UI_ROOT}/execution/{handle.execution_id}")
        for second in range(150):
            status = handle.get_status()
            crm_done = task_completed(handle.execution_id, "notebook_crm_reviewer")
            security_done = task_completed(handle.execution_id, "notebook_security_reviewer")
            billing_active = task_active(handle.execution_id, "check_billing_status")
            print(
                f"[{second:02d}s] status={status.status or 'UNKNOWN'} complete={status.is_complete} "
                f"crm_done={crm_done} security_done={security_done} billing_status_active={billing_active}"
            )
            if crm_done and security_done and billing_active and not status.is_complete:
                print("outage_state = CRM and security completed while billing/provisioning is blocked")
                print("blocked_dependency = billing-api")
                return
            time.sleep(1)
    raise TimeoutError("execution did not reach expected billing outage state")


def resume_execution() -> None:
    execution_id = EXECUTION_FILE.read_text().strip()
    with AgentRuntime(server_url=SERVER_URL) as runtime:
        handle = AgentHandle(execution_id=execution_id, runtime=runtime)
        for second in range(240):
            status = handle.get_status()
            print(f"[{second:02d}s] status={status.status or 'UNKNOWN'} complete={status.is_complete} waiting={status.is_waiting}")
            if status.is_waiting or waiting_human_execution_id(execution_id):
                print("approval = approving paid-plan checkpoint")
                if not approve_waiting_human(runtime, execution_id):
                    handle.approve()
                time.sleep(2)
            if status.is_complete:
                result = handle.join(timeout=5)
                print("final_status =", result.status)
                print("verified = same execution id completed after billing-api recovery")
                return
            time.sleep(1)
    raise TimeoutError("execution did not complete after billing recovery")


def main() -> None:
    command = sys.argv[1] if len(sys.argv) > 1 else "help"
    if command == "deploy":
        deploy()
    elif command == "serve":
        serve()
    elif command == "start":
        start_execution()
    elif command == "resume":
        resume_execution()
    else:
        raise SystemExit("usage: onboarding_system.py [deploy|serve|start|resume]")


if __name__ == "__main__":
    main()
''')

print("wrote", ROOT / "onboarding_system.py")

## 6. Deploy the Agent and Start the Worker

`runtime.deploy(...)` registers the agent definition on the Agentspan server. The worker process keeps the Python tools alive outside Docker so the server can schedule tool work to it.

In [ ]:
PYTHON = sys.executable
DEMO_SCRIPT = ROOT / "onboarding_system.py"
os.environ["AGENTSPAN_UI_ROOT"] = UI_ROOT

sh(f"{PYTHON} {DEMO_SCRIPT} deploy", timeout=180)

worker_log = open(ROOT / "worker.log", "w")
worker_proc = subprocess.Popen(
    [PYTHON, str(DEMO_SCRIPT), "serve"],
    stdout=worker_log,
    stderr=subprocess.STDOUT,
    cwd=str(ROOT),
    env=os.environ.copy(),
)
time.sleep(5)
print("worker_pid =", worker_proc.pid)

## 7. Stop Billing and Start the Execution

This cell creates the failure condition. The billing service is stopped before the run begins.

The expected state is:

- CRM lane completes.
- Security lane completes.
- Billing/provisioning lane remains active on the billing status check.
- The execution is visible in the UI while the dependency is unavailable.

In [ ]:
sh("docker compose stop billing-api", timeout=120)
sh("docker compose ps", timeout=120)

sh(f"{PYTHON} {DEMO_SCRIPT} start", timeout=240)

execution_id = (ROOT / "latest_execution_id.txt").read_text().strip()
execution_url = f"{UI_ROOT}/execution/{execution_id}"
print("execution_id =", execution_id)
display(HTML(f'<p><a href="{execution_url}" target="_blank">Open running parent execution</a></p>'))
display(HTML(f'<iframe src="{execution_url}" width="100%" height="720" style="border:1px solid #ddd;border-radius:8px;"></iframe>'))

## 8. Bring Billing Back and Resume

Now the unavailable dependency returns. The same execution ID continues; the workflow does not create a new parent execution. If the paid-plan tool reaches a human approval checkpoint, the notebook approves it so the demo can finish end to end.

In [ ]:
sh("docker compose up -d billing-api", timeout=180)
sh("docker compose ps", timeout=120)

sh(f"{PYTHON} {DEMO_SCRIPT} resume", timeout=360)

completed_url = f"{UI_ROOT}/execution/{execution_id}"
display(HTML(f'<p><a href="{completed_url}" target="_blank">Open completed parent execution</a></p>'))
display(HTML(f'<iframe src="{completed_url}" width="100%" height="720" style="border:1px solid #ddd;border-radius:8px;"></iframe>'))

## What to Inspect in the UI

Start with the parent execution. It shows the parallel strategy and the top-level status.

Then open the billing/provisioning child execution. That child execution has its own execution ID because it is a child agent lane inside the parent workflow. Seeing two IDs does not mean the run restarted. It means you are moving between parent and child execution views.

The main things to verify:

- The parent execution ID remains stable.
- CRM and security complete before billing recovers.
- Billing/provisioning is blocked while `billing-api` is stopped.
- After `billing-api` comes back, the same execution completes.

## Cleanup

Run this cell when you want to stop the worker and Docker stack.

In [ ]:
try:
    worker_proc.terminate()
    print("worker_stopped =", worker_proc.pid)
except NameError:
    print("worker_proc was not defined")

sh("docker compose down -v --remove-orphans", check=False, timeout=180)